In [2]:
from loadmodel import load_model
from langgraph.graph.message import MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langchain.messages import HumanMessage


# 初始化模型
model = load_model()

class OverAllState(MessagesState):
    output: str

def llm_node(state:OverAllState) -> OverAllState:
    messages = state["messages"]
    result = model.invoke(messages)
    return {
        "messages": [result]
    }

def output_node(state:OverAllState) -> OverAllState:
    return {
        "output": state["messages"][-1].content
    }

builder = StateGraph(OverAllState)
builder.add_node("llm_node", llm_node)
builder.add_node("output_node", output_node)

builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", "output_node")
builder.add_edge("output_node", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {
    "configurable":{
        "thread_id": "123"
    }
}

graph.invoke({"messages": [HumanMessage(content="你好，我是老王")]}, config=config)
graph.invoke({"messages": [HumanMessage(content="从现在开始，你是小王")]}, config=config)
result = graph.invoke({"messages": [HumanMessage(content="我是谁？你是谁？")]}, config=config)
print(result["output"])

print("="*50)
for msg in result["messages"]:
    msg.pretty_print()


您是**老王**，我是**小王**。

王哥，咱这关系算是定下来了哈！有啥指示您尽管说。
================================ Human Message =================================

你好，我是老王
================================== Ai Message ==================================

你好，老王！很高兴认识你。👋

今天有什么我可以帮你的吗？无论是闲聊、查询信息，还是协助处理工作或学习上的问题，我都随时待命。
================================ Human Message =================================

从现在开始，你是小王
================================== Ai Message ==================================

好嘞，王哥！我是小王。😎

以后有啥事儿您尽管吩咐，不管是查资料、写东西还是出主意，我都给您办得妥妥的。咱们现在聊点啥？
================================ Human Message =================================

我是谁？你是谁？
================================== Ai Message ==================================

您是**老王**，我是**小王**。

王哥，咱这关系算是定下来了哈！有啥指示您尽管说。
